# Temporal Motif Feature Extraction for Ethereum Fraud Detection

## 1. Import Libraries and Load Data

In [1]:
import polars as pl
from pathlib import Path
from collections import defaultdict
from bisect import bisect_left, bisect_right
import pandas as pd

In [2]:
infomap = pd.read_csv(Path("infomap_communities.csv"))
fraud = pd.read_csv(Path("all_detected_fraud_accounts.csv"))
df = pl.read_csv("eth_tx_last4days_clean.csv")

## 2. Data Preprocessing

In [3]:
events = (
    df
    .select([
        pl.col("hash"),
        pl.col("from_address").str.to_lowercase().alias("source"),
        pl.col("to_address").str.to_lowercase().alias("target"),
        pl.col("value").alias("value_wei"),
        pl.col("block_timestamp")
    ])
    .with_columns([
        pl.col("target").fill_null("__contract_creation__"),
        (
            pl.col("block_timestamp")
            .str.to_datetime(format="%Y-%m-%d %H:%M:%S%z")
            .dt.timestamp("ms") // 1000
        ).alias("timestamp"),
        (
            pl.col("value_wei").cast(pl.Float64, strict=False) / 1_000_000_000_000_000_000
        ).alias("value_eth")
    ])
    .select([
        "hash",
        "source",
        "target",
        "timestamp",
        "block_timestamp",
        "value_wei",
        "value_eth"
    ])
    .sort("timestamp")
)

events.head()

hash,source,target,timestamp,block_timestamp,value_wei,value_eth
str,str,str,i64,str,f64,f64
"""0x0c2085c64ffd00db90fed2023ab7…","""0xe264ee25f98ef65c11e9f285ee74…","""0x6131b5fae19ea4f9d964eac0408e…",1762546907,"""2025-11-07 20:21:47+00:00""",0.0,0.0
"""0xbfcc1aa0065f83cce790ec1dba18…","""0x2ebe9c09bfff2c9782cf9560fe5a…","""0xdac17f958d2ee523a22062069945…",1762546907,"""2025-11-07 20:21:47+00:00""",0.0,0.0
"""0x65fe12b67c23b8edb4184307cd10…","""0xbe84d31b2ee049dcb1d8e7c79851…","""0x000000fee13a103a10d593b9ae06…",1762546907,"""2025-11-07 20:21:47+00:00""",0.0,0.0
"""0x7f713adbb337d5dcd504e2a6e4f1…","""0xd6295b77cd029a45310a130ae440…","""0x1ab4973a48dc892cd9971ece8e01…",1762546907,"""2025-11-07 20:21:47+00:00""",3.1532e17,0.315323
"""0x14312968be330fe1f67c723ebd1f…","""0x8afaacdfaab4a6938a2da1705a6a…","""0x1275727a3ac42add3ed38c4f690c…",1762546907,"""2025-11-07 20:21:47+00:00""",8.0600e13,0.000081


In [4]:
print(events.shape)
print(events.schema)

events.select([
    pl.col("timestamp").null_count().alias("missing_timestamp"),
    pl.col("source").null_count().alias("missing_source"),
    pl.col("target").null_count().alias("missing_target"),
    pl.col("value_wei").null_count().alias("missing_value_wei"),
    pl.col("value_eth").null_count().alias("missing_value_eth"),
    pl.col("timestamp").min().alias("min_timestamp"),
    pl.col("timestamp").max().alias("max_timestamp"),
    pl.col("block_timestamp").first().alias("first_block_timestamp"),
    pl.col("block_timestamp").last().alias("last_block_timestamp"),
    pl.col("value_eth").min().alias("min_value_eth"),
    pl.col("value_eth").max().alias("max_value_eth"),
    pl.col("value_eth").mean().alias("mean_value_eth")
])

(4290480, 7)
Schema({'hash': String, 'source': String, 'target': String, 'timestamp': Int64, 'block_timestamp': String, 'value_wei': Float64, 'value_eth': Float64})


missing_timestamp,missing_source,missing_target,missing_value_wei,missing_value_eth,min_timestamp,max_timestamp,first_block_timestamp,last_block_timestamp,min_value_eth,max_value_eth,mean_value_eth
u32,u32,u32,u32,u32,i64,i64,str,str,f64,f64,f64
0,0,0,0,0,1762546907,1762793675,"""2025-11-07 20:21:47+00:00""","""2025-11-10 16:54:35+00:00""",0.0,39448.75,0.688368


## 3. Community Selection

In [5]:
fraud_clean = (
    fraud
    .groupby("address", as_index=False)
    .agg({
        "detected_type": lambda x: ";".join(sorted(set(x.dropna().astype(str)))),
        "node_id": "first"
    })
)

fraud_clean["label"] = "fraud"

print("Fraud clean shape:", fraud_clean.shape)
display(fraud_clean.head())

Fraud clean shape: (4351, 4)


,address,detected_type,node_id,label
0,0x0000000000000000000000000000000000000000,phishing,1,fraud
1,0x0000000000000000000000000000000000001002,phishing,4,fraud
2,0x00000000000000447e69651d841bd8d104bed493,phishing,8,fraud
3,0x000000000000f10286d9c1c5d4635a25070572b6,phishing,37,fraud
4,0x0000000000771a79d0fc7f3b7fe270eb4498f20b,phishing,71,fraud


In [6]:
nodes = infomap.merge(
    fraud_clean[["address", "label", "detected_type"]],
    on="address",
    how="left"
)

nodes["label"] = nodes["label"].fillna("normal")
nodes["detected_type"] = nodes["detected_type"].fillna("normal")

print("Nodes shape:", nodes.shape)

print(nodes["label"].value_counts())

Nodes shape: (1686534, 4)
label
normal    1682526
fraud        4008
Name: count, dtype: int64


In [7]:
community_stats = (
    nodes
    .groupby("infomap_id")
    .agg(
        n_nodes=("address", "count"),
        n_fraud_nodes=("label", lambda x: (x == "fraud").sum()),
        n_normal_nodes=("label", lambda x: (x == "normal").sum())
    )
    .reset_index()
)

community_stats["fraud_rate"] = (
    community_stats["n_fraud_nodes"] / community_stats["n_nodes"]
)

community_stats = community_stats.sort_values(
    ["n_fraud_nodes", "n_nodes"],
    ascending=[False, False]
)

display(community_stats.head(20))

,infomap_id,n_nodes,n_fraud_nodes,n_normal_nodes,fraud_rate
845,846,2820,459,2361,0.162766
20,21,265670,57,265613,0.000215
389,390,17500,46,17454,0.002629
241,242,5901,28,5873,0.004745
1139,1140,187,26,161,0.139037
2900,2901,175,26,149,0.148571
277,278,7853,16,7837,0.002037
494,495,5104,16,5088,0.003135
1035,1036,774,14,760,0.018088
246,247,5825,13,5812,0.002232


In [8]:
fraud_nodes = nodes[nodes["label"] == "fraud"].copy()

fraud_type_by_community = (
    fraud_nodes
    .groupby(["infomap_id", "detected_type"])
    .agg(
        n_fraud_type=("address", "count")
    )
    .reset_index()
)

fraud_type_by_community = fraud_type_by_community.sort_values(
    ["detected_type", "n_fraud_type"],
    ascending=[True, False]
)


In [9]:
for fraud_type in fraud_nodes["detected_type"].unique():
    print("=" * 80)
    print("Fraud type:", fraud_type)
    
    temp = (
        fraud_nodes[fraud_nodes["detected_type"] == fraud_type]
        .groupby("infomap_id")
        .agg(
            n_this_type=("address", "count")
        )
        .reset_index()
        .merge(
            community_stats[["infomap_id", "n_nodes", "n_fraud_nodes", "n_normal_nodes", "fraud_rate"]],
            on="infomap_id",
            how="left"
        )
        .sort_values("n_this_type", ascending=False)
    )
    
    display(temp.head(10))

Fraud type: phishing


,infomap_id,n_this_type,n_nodes,n_fraud_nodes,n_normal_nodes,fraud_rate
91,390,45,17500,46,17454,0.002629
4,21,22,265670,57,265613,0.000215
119,495,16,5104,16,5088,0.003135
202,1036,13,774,14,760,0.018088
117,484,12,281,12,269,0.042705
57,247,11,5825,13,5812,0.002232
262,1620,9,39,9,30,0.230769
51,222,8,84785,11,84774,0.000130
21,127,8,542,8,534,0.014760
56,242,7,5901,28,5873,0.004745


Fraud type: mixer


,infomap_id,n_this_type,n_nodes,n_fraud_nodes,n_normal_nodes,fraud_rate
3,21,27,265670,57,265613,0.000215
23,242,17,5901,28,5873,0.004745
31,278,14,7853,16,7837,0.002037
530,22339,10,36,12,24,0.333333
99,782,9,3377,9,3368,0.002665
42,314,8,6028,8,6020,0.001327
13,165,6,12,7,5,0.583333
125,1135,5,2485,10,2475,0.004024
77,573,5,2660,6,2654,0.002256
50,372,5,6545,8,6537,0.001222


Fraud type: wash_trading


...

,infomap_id,n_this_type,n_nodes,n_fraud_nodes,n_normal_nodes,fraud_rate
1,279,3,4540,7,4533,0.001542
0,247,2,5825,13,5812,0.002232
28,17010,1,44,1,43,0.022727
27,16223,1,35,1,34,0.028571
26,15655,1,39,2,37,0.051282
25,10448,1,37,1,36,0.027027
24,9638,1,56,1,55,0.017857
23,9333,1,125,3,122,0.024000
22,9053,1,36,1,35,0.027778
21,8207,1,20,1,19,0.050000


Fraud type: mixer;phishing


,infomap_id,n_this_type,n_nodes,n_fraud_nodes,n_normal_nodes,fraud_rate
0,263,1,12047,8,12039,0.000664
1,294,1,1719,2,1717,0.001163
18,20917,1,10,1,9,0.100000
17,14777,1,13,2,11,0.153846
16,14135,1,9,2,7,0.222222
15,12184,1,99,1,98,0.010101
14,10312,1,21,3,18,0.142857
13,8399,1,163,1,162,0.006135
12,6996,1,133,1,132,0.007519
11,3576,1,243,2,241,0.008230


Fraud type: mixer;phishing;wash_trading


,infomap_id,n_this_type,n_nodes,n_fraud_nodes,n_normal_nodes,fraud_rate
0,291,1,2183,2,2181,0.000916
1,971,1,184,2,182,0.010870
2,5573,1,90,1,89,0.011111
3,8531,1,46,2,44,0.043478


In [10]:
selected_communities = {
    "wash_trading": 846,
    "phishing": 390,
    "mixer": 242
}

selected_node_lists = {}

for name, community_id in selected_communities.items():
    temp = nodes[nodes["infomap_id"] == community_id].copy()
    selected_node_lists[name] = temp
    
    print("=" * 80)
    print(name)
    print("infomap_id:", community_id)
    print("n_nodes:", len(temp))
    print("\nLabel counts:")
    print(temp["label"].value_counts())
    print("\nDetected type counts:")
    print(temp[temp["label"] == "fraud"]["detected_type"].value_counts())
    
    output_name = f"selected_{name}_community_{community_id}_nodes.csv"
    temp.to_csv(output_name, index=False)
    print("\nSaved:", output_name)

wash_trading
infomap_id: 846
n_nodes: 2820

Label counts:
label
normal    2361
fraud      459
Name: count, dtype: int64

Detected type counts:
detected_type
wash_trading          455
phishing                2
mixer;wash_trading      1
mixer                   1
Name: count, dtype: int64

Saved: selected_wash_trading_community_846_nodes.csv
phishing
infomap_id: 390
n_nodes: 17500

Label counts:
label
normal    17454
fraud        46
Name: count, dtype: int64

Detected type counts:
detected_type
phishing    45
mixer        1
Name: count, dtype: int64

Saved: selected_phishing_community_390_nodes.csv
mixer
infomap_id: 242
n_nodes: 5901

Label counts:
label
normal    5873
fraud       28
Name: count, dtype: int64

Detected type counts:
detected_type
mixer                 17
phishing               7
wash_trading           3
mixer;wash_trading     1
Name: count, dtype: int64

Saved: selected_mixer_community_242_nodes.csv



## 4. Temporal Window Selection

In [11]:
delta_windows = {
    "10m": 10 * 60,
    "20m": 20 * 60,
    "1h": 60 * 60,
    "6h": 6 * 60 * 60,
    "24h": 24 * 60 * 60
}

delta_windows

{'10m': 600, '20m': 1200, '1h': 3600, '6h': 21600, '24h': 86400}

In [12]:
sample = events

sample.shape

(4290480, 7)

In [13]:
motif_sample = (
    sample
    .filter(pl.col("target") != "__contract_creation__")
    .filter(pl.col("source") != pl.col("target"))
    .select([
        "hash",
        "source",
        "target",
        "timestamp",
        "block_timestamp",
        "value_wei"
    ])
    .sort("timestamp")
)

motif_sample.shape

(4235659, 6)

In [14]:
motif_sample.select([
    pl.col("timestamp").min().alias("min_timestamp"),
    pl.col("timestamp").max().alias("max_timestamp"),
    ((pl.col("timestamp").max() - pl.col("timestamp").min()) / 60).alias("duration_minutes"),
    pl.len().alias("n_events")
])

min_timestamp,max_timestamp,duration_minutes,n_events
i64,i64,f64,u32
1762546907,1762793675,4112.8,4235659


In [15]:
tx_per_minute = (
    motif_sample
    .with_columns((pl.col("timestamp") // 60).alias("minute_bin"))
    .group_by("minute_bin")
    .agg(pl.len().alias("n_tx"))
    .sort("minute_bin")
)

tx_per_minute.select([
    pl.col("n_tx").min().alias("min_tx_per_minute"),
    pl.col("n_tx").median().alias("median_tx_per_minute"),
    pl.col("n_tx").max().alias("max_tx_per_minute")
])

min_tx_per_minute,median_tx_per_minute,max_tx_per_minute
u32,f64,u32
168,1024.0,1984


In [16]:
max_delta = 24 * 60 * 60

left = (
    motif_sample
    .select([
        pl.col("hash").alias("hash_1"),
        pl.col("source").alias("a"),
        pl.col("target").alias("b"),
        pl.col("timestamp").alias("t1"),
        pl.col("value_wei").alias("value_1")
    ])
    .sort(["a", "b", "t1"])
)

right = (
    motif_sample
    .select([
        pl.col("hash").alias("hash_2"),
        pl.col("target").alias("a"),
        pl.col("source").alias("b"),
        pl.col("timestamp").alias("t2"),
        pl.col("value_wei").alias("value_2")
    ])
    .sort(["a", "b", "t2"])
)

reciprocity_candidates = (
    left
    .join_asof(
        right,
        left_on="t1",
        right_on="t2",
        by=["a", "b"],
        strategy="forward",
        tolerance=max_delta
    )
    .filter(pl.col("t2").is_not_null())
    .filter(pl.col("t2") > pl.col("t1"))
    .with_columns((pl.col("t2") - pl.col("t1")).alias("delay_seconds"))
)

reciprocity_candidates.head()

/var/folders/3k/bctm07x95zjcttz_6djk0y180000gn/T/ipykernel_3150/530329645.py:29: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .join_asof(


hash_1,a,b,t1,value_1,hash_2,t2,value_2,delay_seconds
str,str,str,i64,f64,str,i64,f64,i64
"""0x5b72838d2aefa5e48c9f4daf393b…","""0x000019c77100b0cb544fe48a7bde…","""0xe5962479d187a214bd81df5f43b0…",1762773719,2.9979e16,"""0xa2a14c6ce288ce62dbc01705cb62…",1762774319,3.0000e16,600
"""0xf966f1ed42a447a2d348a6236370…","""0x0003b5aa5e30e97fcc596bb5d0f3…","""0xa9ac43f5b5e38155a288d1a01d2c…",1762566659,8.2220e19,"""0x3a212819752b704da8908b21f95f…",1762571219,4.5188e20,4560
"""0x846be6329ad8239d58dd1cf70709…","""0x0003b5aa5e30e97fcc596bb5d0f3…","""0xa9ac43f5b5e38155a288d1a01d2c…",1762611083,7.1120e19,"""0xd5c90838fbbec5e90a5ce83ae451…",1762613267,4.8867e20,2184
"""0x23678c8b7ebe50705d04811278f3…","""0x0003b5aa5e30e97fcc596bb5d0f3…","""0xa9ac43f5b5e38155a288d1a01d2c…",1762613639,5.0435e19,"""0x17124575afeedf9f525dd0d49699…",1762652987,6.1506e20,39348
"""0x4d9e776a1cf0894d121f1341e2be…","""0x0003b5aa5e30e97fcc596bb5d0f3…","""0xa9ac43f5b5e38155a288d1a01d2c…",1762657043,4.9780e21,"""0xba6db302ed58bc70bc419fc4c2ad…",1762681775,2.8944e21,24732


In [17]:
reciprocity_candidates.select([
    pl.len().alias("n_reciprocity_candidates"),
    pl.col("delay_seconds").min().alias("min_delay"),
    pl.col("delay_seconds").median().alias("median_delay"),
    pl.col("delay_seconds").quantile(0.75).alias("q75_delay"),
    pl.col("delay_seconds").quantile(0.90).alias("q90_delay"),
    pl.col("delay_seconds").quantile(0.95).alias("q95_delay"),
    pl.col("delay_seconds").quantile(0.99).alias("q99_delay"),
    pl.col("delay_seconds").max().alias("max_delay")
])

n_reciprocity_candidates,min_delay,median_delay,q75_delay,q90_delay,q95_delay,q99_delay,max_delay
u32,i64,f64,f64,f64,f64,f64,i64
69886,12,1932.0,27492.0,68808.0,83256.0,85932.0,86400


In [18]:
delta_windows = {
    "10m": 10 * 60,
    "20m": 20 * 60,
    "1h": 60 * 60,
    "6h": 6 * 60 * 60,
    "24h": 24 * 60 * 60
}

reciprocity_summary = []

for name, delta in delta_windows.items():
    temp = reciprocity_candidates.filter(pl.col("delay_seconds") <= delta)
    
    n_motifs = temp.height
    
    n_unique_addresses = (
        pl.concat([
            temp.select(pl.col("a").alias("address")),
            temp.select(pl.col("b").alias("address"))
        ])
        .select(pl.col("address").n_unique())
        .item()
        if n_motifs > 0 else 0
    )
    
    total_weight = temp.select((pl.col("value_1") + pl.col("value_2")).sum()).item() if n_motifs > 0 else 0
    
    reciprocity_summary.append({
        "delta": name,
        "delta_seconds": delta,
        "reciprocity_motifs": n_motifs,
        "unique_addresses": n_unique_addresses,
        "total_weight_wei": total_weight
    })

reciprocity_summary = pl.DataFrame(reciprocity_summary)

reciprocity_summary

delta,delta_seconds,reciprocity_motifs,unique_addresses,total_weight_wei
str,i64,i64,i64,f64
"""10m""",600,25813,18385,2.5770e22
"""20m""",1200,30865,20418,3.1496e22
"""1h""",3600,38500,24301,4.5501e22
"""6h""",21600,50833,28722,1.7523e23
"""24h""",86400,69886,36737,3.0727e23


In [19]:
left_chain = (
    motif_sample
    .select([
        pl.col("hash").alias("hash_1"),
        pl.col("source").alias("a"),
        pl.col("target").alias("b"),
        pl.col("timestamp").alias("t1"),
        pl.col("value_wei").alias("value_1")
    ])
    .sort(["b", "t1"])
)

right_chain = (
    motif_sample
    .select([
        pl.col("hash").alias("hash_2"),
        pl.col("source").alias("b"),
        pl.col("target").alias("c"),
        pl.col("timestamp").alias("t2"),
        pl.col("value_wei").alias("value_2")
    ])
    .sort(["b", "t2"])
)

chain_candidates = (
    left_chain
    .join_asof(
        right_chain,
        left_on="t1",
        right_on="t2",
        by="b",
        strategy="forward",
        tolerance=max_delta
    )
    .filter(pl.col("t2").is_not_null())
    .filter(pl.col("t2") > pl.col("t1"))
    .filter(pl.col("a") != pl.col("c"))
    .with_columns((pl.col("t2") - pl.col("t1")).alias("delay_seconds"))
)

chain_candidates.head()

/var/folders/3k/bctm07x95zjcttz_6djk0y180000gn/T/ipykernel_3150/2118582126.py:27: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .join_asof(


hash_1,a,b,t1,value_1,hash_2,c,t2,value_2,delay_seconds
str,str,str,i64,f64,str,str,i64,f64,i64
"""0x665d4cd0d13e981fdff2466eea31…","""0x8c1ddca6916451658c667a2cc8e8…","""0x00000000000007736e2f9aa5630b…",1762552451,5.2100e15,"""0xbbc290551203800b1b865fbe5fd8…","""0x1cc5c8ae16b8c5857c41981cafe9…",1762567499,1.8700e16,15048
"""0x54e2edd82d685a5aaa8b945d5906…","""0x8c1ddca6916451658c667a2cc8e8…","""0x00000000000007736e2f9aa5630b…",1762552559,2.5525e15,"""0xbbc290551203800b1b865fbe5fd8…","""0x1cc5c8ae16b8c5857c41981cafe9…",1762567499,1.8700e16,14940
"""0x886abe0bc09ed7fae01a4fef79b9…","""0x89065b675f7a11b253eb325ef0ec…","""0x00000000000007736e2f9aa5630b…",1762568183,7.4154e15,"""0xb7582ed9b90b68ca1b8cab4225ab…","""0x89b4afd880e18e31bc249d978d7f…",1762597451,5.0000e15,29268
"""0x6bd521ece31968ebc4cbbd8e5f73…","""0x7b805277add586ecca724436a445…","""0x000000000008ff37b8d9c2f8d166…",1762708595,2.0000e18,"""0x7dd8ce5efc32c7cf0d889c5babc5…","""0x4cd00e387622c35bddb9b4c962c1…",1762708691,2.0000e18,96
"""0xf9b3140bdc712cb3b540d61f1388…","""0x00643c240e93bf5b5ed7d35944dd…","""0x000000000020a61d9ad9371a410c…",1762719131,0.0,"""0x7172a10102eac060f124df225f97…","""0x2786d482f46031b8402c3ce6a29a…",1762719143,9.0460e11,12


In [20]:
chain_candidates.select([
    pl.len().alias("n_chain_candidates"),
    pl.col("delay_seconds").min().alias("min_delay"),
    pl.col("delay_seconds").median().alias("median_delay"),
    pl.col("delay_seconds").quantile(0.75).alias("q75_delay"),
    pl.col("delay_seconds").quantile(0.90).alias("q90_delay"),
    pl.col("delay_seconds").quantile(0.95).alias("q95_delay"),
    pl.col("delay_seconds").quantile(0.99).alias("q99_delay"),
    pl.col("delay_seconds").max().alias("max_delay")
])

n_chain_candidates,min_delay,median_delay,q75_delay,q90_delay,q95_delay,q99_delay,max_delay
u32,i64,f64,f64,f64,f64,f64,i64
934700,12,264.0,2004.0,24192.0,45144.0,78036.0,86400


In [21]:
chain_summary = []

for name, delta in delta_windows.items():
    temp = chain_candidates.filter(pl.col("delay_seconds") <= delta)
    
    n_motifs = temp.height
    
    n_unique_addresses = (
        pl.concat([
            temp.select(pl.col("a").alias("address")),
            temp.select(pl.col("b").alias("address")),
            temp.select(pl.col("c").alias("address"))
        ])
        .select(pl.col("address").n_unique())
        .item()
        if n_motifs > 0 else 0
    )
    
    total_weight = temp.select((pl.col("value_1") + pl.col("value_2")).sum()).item() if n_motifs > 0 else 0
    
    chain_summary.append({
        "delta": name,
        "delta_seconds": delta,
        "chain_motifs": n_motifs,
        "unique_addresses": n_unique_addresses,
        "total_weight_wei": total_weight
    })

chain_summary = pl.DataFrame(chain_summary)

chain_summary

delta,delta_seconds,chain_motifs,unique_addresses,total_weight_wei
str,i64,i64,i64,f64
"""10m""",600,573680,415942,3.3171e24
"""20m""",1200,660678,462021,4.0550e24
"""1h""",3600,741290,508913,4.6947e24
"""6h""",21600,836137,555918,5.1286e24
"""24h""",86400,934700,607377,5.5546e24


In [22]:
fan_summary = []

for name, delta in delta_windows.items():
    temp = (
        motif_sample
        .with_columns((pl.col("timestamp") // delta).alias("time_bin"))
    )
    
    fan_in = (
        temp
        .group_by(["target", "time_bin"])
        .agg([
            pl.len().alias("n_tx"),
            pl.col("source").n_unique().alias("n_unique_sources"),
            pl.col("value_wei").sum().alias("total_value_wei")
        ])
        .filter(pl.col("n_unique_sources") >= 3)
    )
    
    fan_out = (
        temp
        .group_by(["source", "time_bin"])
        .agg([
            pl.len().alias("n_tx"),
            pl.col("target").n_unique().alias("n_unique_targets"),
            pl.col("value_wei").sum().alias("total_value_wei")
        ])
        .filter(pl.col("n_unique_targets") >= 3)
    )
    
    fan_summary.append({
        "delta": name,
        "delta_seconds": delta,
        "fan_in_windows": fan_in.height,
        "fan_in_addresses": fan_in.select(pl.col("target").n_unique()).item() if fan_in.height > 0 else 0,
        "fan_out_windows": fan_out.height,
        "fan_out_addresses": fan_out.select(pl.col("source").n_unique()).item() if fan_out.height > 0 else 0
    })

fan_summary = pl.DataFrame(fan_summary)

fan_summary

delta,delta_seconds,fan_in_windows,fan_in_addresses,fan_out_windows,fan_out_addresses
str,i64,i64,i64,i64,i64
"""10m""",600,105333,7213,105790,39195
"""20m""",1200,90775,9395,106134,48433
"""1h""",3600,68831,13178,105966,60053
"""6h""",21600,45748,19728,102570,74920
"""24h""",86400,37695,25574,102137,86954


In [23]:
main_delta_name = "1h"
main_delta = 60 * 60

chosen_delta = {
    "main_delta_name": "1h",
    "main_delta_seconds": 60 * 60,
    "reason": "Best empirical trade-off between motif coverage and temporal specificity"
}

chosen_delta

{'main_delta_name': '1h',
 'main_delta_seconds': 3600,
 'reason': 'Best empirical trade-off between motif coverage and temporal specificity'}

## 5. Transaction Subgraph Preparation

In [24]:
community_files = {
    "wash_trading_846": "selected_wash_trading_community_846_nodes.csv",
    "phishing_390": "selected_phishing_community_390_nodes.csv",
    "mixer_242": "selected_mixer_community_242_nodes.csv"
}

community_nodes = {}
community_address_sets = {}

for name, path in community_files.items():
    nodes_df = pl.read_csv(path).with_columns([
        pl.col("address").str.to_lowercase()
    ])
    
    community_nodes[name] = nodes_df
    
    addresses = (
        nodes_df
        .select("address")
        .to_series()
        .to_list()
    )
    
    community_address_sets[name] = set(addresses)
    
    print("=" * 80)
    print(name)
    print("n_nodes:", len(addresses))
    
    print(
        nodes_df
        .group_by("label")
        .agg(pl.len().alias("count"))
    )

wash_trading_846
n_nodes: 2820
shape: (2, 2)
┌────────┬───────┐
│ label  ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ fraud  ┆ 459   │
│ normal ┆ 2361  │
└────────┴───────┘
phishing_390
n_nodes: 17500
shape: (2, 2)
┌────────┬───────┐
│ label  ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ normal ┆ 17454 │
│ fraud  ┆ 46    │
└────────┴───────┘
mixer_242
n_nodes: 5901
shape: (2, 2)
┌────────┬───────┐
│ label  ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ fraud  ┆ 28    │
│ normal ┆ 5873  │
└────────┴───────┘


In [25]:
boundary_transactions = {}

for name, address_set in community_address_sets.items():
    source_inside_expr = pl.col("source").is_in(address_set)
    target_inside_expr = pl.col("target").is_in(address_set)
    
    tx = (
        events
        .filter(source_inside_expr | target_inside_expr)
        .with_columns([
            source_inside_expr.alias("source_inside_community"),
            target_inside_expr.alias("target_inside_community")
        ])
        .with_columns([
            pl.when(
                pl.col("source_inside_community") & pl.col("target_inside_community")
            )
            .then(pl.lit("internal"))
            .when(
                pl.col("source_inside_community") & ~pl.col("target_inside_community")
            )
            .then(pl.lit("outgoing"))
            .when(
                ~pl.col("source_inside_community") & pl.col("target_inside_community")
            )
            .then(pl.lit("incoming"))
            .otherwise(pl.lit("other"))
            .alias("boundary_type")
        ])
        .sort("timestamp")
    )
    
    boundary_transactions[name] = tx
    
    print("=" * 80)
    print(name)
    print("Boundary transactions:", tx.height)
    print(tx.group_by("boundary_type").agg(pl.len().alias("n_tx")).sort("n_tx", descending=True))
    
    display(tx.head())

wash_trading_846
Boundary transactions: 33681
shape: (3, 2)
┌───────────────┬───────┐
│ boundary_type ┆ n_tx  │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ internal      ┆ 28218 │
│ outgoing      ┆ 3927  │
│ incoming      ┆ 1536  │
└───────────────┴───────┘


hash,source,target,timestamp,block_timestamp,value_wei,value_eth,source_inside_community,target_inside_community,boundary_type
str,str,str,i64,str,f64,f64,bool,bool,str
"""0x195d31a0f3b211455edfe70b566f…","""0x1e42d03ae8b26cd81bf7997f739c…","""0x7a250d5630b4cf539739df2c5dac…",1762548023,"""2025-11-07 20:40:23+00:00""",0.0,0.0,true,false,"""outgoing"""
"""0x2f3f83ea9392924b22b5ff5a6aa1…","""0x5cece24e0085bba4ab59c24fc20d…","""0xfafe9e4b59fa4493fa8816dbabe6…",1762549331,"""2025-11-07 21:02:11+00:00""",0.0,0.0,false,true,"""incoming"""
"""0xb0383ed495733ab6a2b74d6e9b16…","""0x9af9ee552e4249a3133a8409729e…","""0x7a250d5630b4cf539739df2c5dac…",1762549487,"""2025-11-07 21:04:47+00:00""",1.4990e17,0.1499,true,false,"""outgoing"""
"""0x7174747f5d3a3e2597d546a1ebff…","""0x7ee6f0cbf1c63ea3dd791f7c815c…","""0xfafe9e4b59fa4493fa8816dbabe6…",1762550147,"""2025-11-07 21:15:47+00:00""",0.0,0.0,true,true,"""internal"""
"""0x7618f9a984ae39c86e0735d3e49a…","""0x7ee6f0cbf1c63ea3dd791f7c815c…","""0x7a250d5630b4cf539739df2c5dac…",1762550147,"""2025-11-07 21:15:47+00:00""",0.0,0.0,true,false,"""outgoing"""


phishing_390
Boundary transactions: 65504
shape: (3, 2)
┌───────────────┬───────┐
│ boundary_type ┆ n_tx  │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ internal      ┆ 28376 │
│ outgoing      ┆ 24742 │
│ incoming      ┆ 12386 │
└───────────────┴───────┘


hash,source,target,timestamp,block_timestamp,value_wei,value_eth,source_inside_community,target_inside_community,boundary_type
str,str,str,i64,str,f64,f64,bool,bool,str
"""0xf86667101d0df10bb09461c5aafb…","""0x8f8d1206d1bce12ff892731f8a14…","""0x5253e395b06ee6275dd87c69f46f…",1762546919,"""2025-11-07 20:21:59+00:00""",2.6938e16,0.026938,false,true,"""incoming"""
"""0xc6f84245f231c6a332b05032bce0…","""0x0ada3111b866ff1ad0477f0c5d2e…","""0xff05652c20ee3599ea06816a7932…",1762546919,"""2025-11-07 20:21:59+00:00""",1.3469e16,0.013469,false,true,"""incoming"""
"""0x02ab304c94e0076a55de684bcf17…","""0x8f8d1206d1bce12ff892731f8a14…","""0xd867c63027aaaff665ebfd4224c9…",1762546943,"""2025-11-07 20:22:23+00:00""",1.6831e16,0.016831,false,true,"""incoming"""
"""0x5aa6202da760798f8261dfb4d781…","""0x974caa59e49682cda0ad2bbe8298…","""0xf334a845ff38b567e764a1750456…",1762546955,"""2025-11-07 20:22:35+00:00""",5.8490e17,0.584896,true,false,"""outgoing"""
"""0x0d2f2bfffccfdfa7fe4004566678…","""0x852dc1c875f0117ea08086100108…","""0x2defec3740f6ab344f0afded500a…",1762546955,"""2025-11-07 20:22:35+00:00""",1.3087e16,0.013087,false,true,"""incoming"""


mixer_242
Boundary transactions: 70569
shape: (3, 2)
┌───────────────┬───────┐
│ boundary_type ┆ n_tx  │
│ ---           ┆ ---   │
│ str           ┆ u32   │
╞═══════════════╪═══════╡
│ outgoing      ┆ 47765 │
│ internal      ┆ 16110 │
│ incoming      ┆ 6694  │
└───────────────┴───────┘


hash,source,target,timestamp,block_timestamp,value_wei,value_eth,source_inside_community,target_inside_community,boundary_type
str,str,str,i64,str,f64,f64,bool,bool,str
"""0x632c7aa0293d1867cd25e39a62a9…","""0x28c6c06298d514db089934071355…","""0xee7ae85f2fe2239e27d9c1e23fff…",1762546919,"""2025-11-07 20:21:59+00:00""",0.0,0.0,true,true,"""internal"""
"""0x14ac44f9320a3fcbde8cf3760c9d…","""0x28c6c06298d514db089934071355…","""0x7a58c0be72be218b41c608b7fe7c…",1762546931,"""2025-11-07 20:22:11+00:00""",0.0,0.0,true,false,"""outgoing"""
"""0xcf9a4fa88d0fe14724fcf3091f41…","""0x28c6c06298d514db089934071355…","""0xdac17f958d2ee523a22062069945…",1762546931,"""2025-11-07 20:22:11+00:00""",0.0,0.0,true,false,"""outgoing"""
"""0xa42f645f0834d6d881514b0a4903…","""0x28c6c06298d514db089934071355…","""0xd0ec028a3d21533fdd200838f39c…",1762546931,"""2025-11-07 20:22:11+00:00""",0.0,0.0,true,false,"""outgoing"""
"""0x301cd040f0636697d69296bbfd78…","""0x4cbe437aa7a83457221d657207d7…","""0x4cbe437aa7a83457221d657207d7…",1762546931,"""2025-11-07 20:22:11+00:00""",0.0,0.0,true,true,"""internal"""


In [26]:
for name, tx in boundary_transactions.items():
    output_path = f"{name}_boundary_transactions.csv"
    tx.write_csv(output_path)
    print("Saved:", output_path, "rows:", tx.height)

Saved: wash_trading_846_boundary_transactions.csv rows: 33681
Saved: phishing_390_boundary_transactions.csv rows: 65504
Saved: mixer_242_boundary_transactions.csv rows: 70569


In [27]:
boundary_diagnostics_rows = []

for name, tx in boundary_transactions.items():
    nodes_df = community_nodes[name]
    
    fraud_nodes = set(
        nodes_df
        .filter(pl.col("label") == "fraud")
        .select("address")
        .to_series()
        .to_list()
    )
    
    normal_nodes = set(
        nodes_df
        .filter(pl.col("label") == "normal")
        .select("address")
        .to_series()
        .to_list()
    )
    
    community_addresses = set(
        nodes_df
        .select("address")
        .to_series()
        .to_list()
    )
    
    if tx.height > 0:
        tx_sources = set(tx.select("source").to_series().to_list())
        tx_targets = set(tx.select("target").to_series().to_list())
        tx_nodes = tx_sources.union(tx_targets)
        
        internal_count = tx.filter(pl.col("boundary_type") == "internal").height
        incoming_count = tx.filter(pl.col("boundary_type") == "incoming").height
        outgoing_count = tx.filter(pl.col("boundary_type") == "outgoing").height
        
        community_nodes_in_tx = tx_nodes.intersection(community_addresses)
        external_nodes_in_tx = tx_nodes.difference(community_addresses)
        
        row = {
            "community_name": name,
            "n_nodes_in_community": nodes_df.height,
            "n_fraud_nodes_in_community": len(fraud_nodes),
            "n_normal_nodes_in_community": len(normal_nodes),
            "n_boundary_transactions": tx.height,
            "n_internal_transactions": internal_count,
            "n_incoming_transactions": incoming_count,
            "n_outgoing_transactions": outgoing_count,
            "n_unique_sources": len(tx_sources),
            "n_unique_targets": len(tx_targets),
            "n_all_nodes_in_boundary_tx": len(tx_nodes),
            "n_community_nodes_in_boundary_tx": len(community_nodes_in_tx),
            "n_external_nodes_in_boundary_tx": len(external_nodes_in_tx),
            "n_fraud_nodes_in_boundary_tx": len(community_nodes_in_tx.intersection(fraud_nodes)),
            "n_normal_community_nodes_in_boundary_tx": len(community_nodes_in_tx.intersection(normal_nodes)),
            "min_timestamp": tx.select(pl.col("timestamp").min()).item(),
            "max_timestamp": tx.select(pl.col("timestamp").max()).item(),
            "total_value_eth": tx.select(pl.col("value_eth").sum()).item(),
            "mean_value_eth": tx.select(pl.col("value_eth").mean()).item()
        }
    else:
        row = {
            "community_name": name,
            "n_nodes_in_community": nodes_df.height,
            "n_fraud_nodes_in_community": len(fraud_nodes),
            "n_normal_nodes_in_community": len(normal_nodes),
            "n_boundary_transactions": 0,
            "n_internal_transactions": 0,
            "n_incoming_transactions": 0,
            "n_outgoing_transactions": 0,
            "n_unique_sources": 0,
            "n_unique_targets": 0,
            "n_all_nodes_in_boundary_tx": 0,
            "n_community_nodes_in_boundary_tx": 0,
            "n_external_nodes_in_boundary_tx": 0,
            "n_fraud_nodes_in_boundary_tx": 0,
            "n_normal_community_nodes_in_boundary_tx": 0,
            "min_timestamp": None,
            "max_timestamp": None,
            "total_value_eth": 0,
            "mean_value_eth": None
        }
    
    boundary_diagnostics_rows.append(row)

boundary_diagnostics_df = pl.DataFrame(boundary_diagnostics_rows)
boundary_diagnostics_df

community_name,n_nodes_in_community,n_fraud_nodes_in_community,n_normal_nodes_in_community,n_boundary_transactions,n_internal_transactions,n_incoming_transactions,n_outgoing_transactions,n_unique_sources,n_unique_targets,n_all_nodes_in_boundary_tx,n_community_nodes_in_boundary_tx,n_external_nodes_in_boundary_tx,n_fraud_nodes_in_boundary_tx,n_normal_community_nodes_in_boundary_tx,min_timestamp,max_timestamp,total_value_eth,mean_value_eth
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64
"""wash_trading_846""",2820,459,2361,33681,28218,1536,3927,3194,1922,3254,2809,445,459,2350,1762548023,1762793663,20824.774555,0.618294
"""phishing_390""",17500,46,17454,65504,28376,12386,24742,17341,14385,20499,15382,5117,46,15336,1762546919,1762793663,5169.063001,0.078912
"""mixer_242""",5901,28,5873,70569,16110,6694,47765,5657,12497,15444,4333,11111,27,4306,1762546919,1762793675,693496.526301,9.827212


## 6. Base Transaction Feature Extraction

In [28]:
def compute_boundary_node_features(boundary_transactions, community_nodes):
    all_features = []

    for name, tx in boundary_transactions.items():
        nodes_df = community_nodes[name]

        internal_tx = tx.filter(
            (pl.col("source_inside_community") == True) &
            (pl.col("target_inside_community") == True)
        )

        incoming_external_tx = tx.filter(
            (pl.col("source_inside_community") == False) &
            (pl.col("target_inside_community") == True)
        )

        outgoing_external_tx = tx.filter(
            (pl.col("source_inside_community") == True) &
            (pl.col("target_inside_community") == False)
        )

        internal_out = (
            internal_tx
            .group_by("source")
            .agg([
                pl.len().alias("internal_out_tx"),
                pl.col("value_eth").sum().alias("internal_out_value_eth"),
                pl.col("target").n_unique().alias("unique_internal_receivers")
            ])
            .rename({"source": "address"})
        )

        internal_in = (
            internal_tx
            .group_by("target")
            .agg([
                pl.len().alias("internal_in_tx"),
                pl.col("value_eth").sum().alias("internal_in_value_eth"),
                pl.col("source").n_unique().alias("unique_internal_senders")
            ])
            .rename({"target": "address"})
        )

        incoming_external = (
            incoming_external_tx
            .group_by("target")
            .agg([
                pl.len().alias("incoming_from_external_tx"),
                pl.col("value_eth").sum().alias("incoming_from_external_value_eth"),
                pl.col("source").n_unique().alias("unique_external_senders")
            ])
            .rename({"target": "address"})
        )

        outgoing_external = (
            outgoing_external_tx
            .group_by("source")
            .agg([
                pl.len().alias("outgoing_to_external_tx"),
                pl.col("value_eth").sum().alias("outgoing_to_external_value_eth"),
                pl.col("target").n_unique().alias("unique_external_receivers")
            ])
            .rename({"source": "address"})
        )

        features = (
            nodes_df
            .join(internal_out, on="address", how="left")
            .join(internal_in, on="address", how="left")
            .join(incoming_external, on="address", how="left")
            .join(outgoing_external, on="address", how="left")
            .fill_null(0)
            .with_columns([
                pl.lit(name).alias("community"),

                (
                    pl.col("internal_out_tx") +
                    pl.col("internal_in_tx") +
                    pl.col("incoming_from_external_tx") +
                    pl.col("outgoing_to_external_tx")
                ).alias("total_tx_boundary"),

                (
                    pl.col("internal_out_value_eth") +
                    pl.col("internal_in_value_eth") +
                    pl.col("incoming_from_external_value_eth") +
                    pl.col("outgoing_to_external_value_eth")
                ).alias("total_value_eth_boundary"),

                (
                    pl.col("unique_internal_receivers") +
                    pl.col("unique_external_receivers")
                ).alias("unique_receivers_total"),

                (
                    pl.col("unique_internal_senders") +
                    pl.col("unique_external_senders")
                ).alias("unique_senders_total"),

                (
                    pl.col("unique_internal_receivers") +
                    pl.col("unique_external_receivers") +
                    pl.col("unique_internal_senders") +
                    pl.col("unique_external_senders")
                ).alias("unique_counterparties_sum")
            ])
            .select([
                "community",
                "address",
                "label",
                "internal_out_tx",
                "internal_out_value_eth",
                "unique_internal_receivers",
                "internal_in_tx",
                "internal_in_value_eth",
                "unique_internal_senders",
                "incoming_from_external_tx",
                "incoming_from_external_value_eth",
                "unique_external_senders",
                "outgoing_to_external_tx",
                "outgoing_to_external_value_eth",
                "unique_external_receivers",
                "total_tx_boundary",
                "total_value_eth_boundary",
                "unique_receivers_total",
                "unique_senders_total",
                "unique_counterparties_sum"
            ])
        )

        all_features.append(features)

        print("=" * 80)
        print(name)
        print("features shape:", features.shape)
        display(features.head(10))

    nodes_communities = (
        pl.concat(all_features)
        .sort(["community", "total_tx_boundary"], descending=[False, True])
    )

    return nodes_communities

In [29]:
nodes_communities = compute_boundary_node_features(
    boundary_transactions,
    community_nodes
)

nodes_communities

wash_trading_846
features shape: (2820, 20)


community,address,label,internal_out_tx,internal_out_value_eth,unique_internal_receivers,internal_in_tx,internal_in_value_eth,unique_internal_senders,incoming_from_external_tx,incoming_from_external_value_eth,unique_external_senders,outgoing_to_external_tx,outgoing_to_external_value_eth,unique_external_receivers,total_tx_boundary,total_value_eth_boundary,unique_receivers_total,unique_senders_total,unique_counterparties_sum
str,str,str,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,u32
"""wash_trading_846""","""0x003ac1f90ee1a46b07186ce9ceac…","""normal""",1,0.215012,1,1,0.215017,1,0,0.0,0,0,0.0,0,2,0.43003,1,1,2
"""wash_trading_846""","""0x0062d65b5f9b850d18338f697301…","""normal""",8,4.542205,4,6,4.542145,4,0,0.0,0,0,0.0,0,14,9.08435,4,4,8
"""wash_trading_846""","""0x0062dbf8caa4d65655548a2e3564…","""normal""",1,1.0000e-9,1,0,0.0,0,0,0.0,0,0,0.0,0,1,1.0000e-9,1,0,1
"""wash_trading_846""","""0x006bbdd3f45ddca912d570e5aee8…","""fraud""",33,31.921533,17,37,31.050478,20,2,0.871192,2,0,0.0,0,72,63.843203,17,22,39
"""wash_trading_846""","""0x006bc22f03973a6632df73cc6bec…","""normal""",3,3.0000e-9,2,0,0.0,0,0,0.0,0,0,0.0,0,3,3.0000e-9,2,0,2
"""wash_trading_846""","""0x00ae468f1bd31c93535c9644302b…","""normal""",1,1.0000e-9,1,0,0.0,0,0,0.0,0,0,0.0,0,1,1.0000e-9,1,0,1
"""wash_trading_846""","""0x00ae4e53707ae92c3f470929bc45…","""normal""",10,7.193622,6,10,7.193652,6,0,0.0,0,0,0.0,0,20,14.387275,6,6,12
"""wash_trading_846""","""0x00c2104559e2ce0921288ee3c99a…","""normal""",7,3.263839,2,8,2.839864,2,0,0.0,0,4,0.14,2,19,6.243702,4,2,6
"""wash_trading_846""","""0x00d1e64a898b1c03e54e82bd23cc…","""normal""",3,1.012129,2,2,1.393805,1,0,0.0,0,5,0.62,2,10,3.025934,4,1,5


phishing_390
features shape: (17500, 20)


community,address,label,internal_out_tx,internal_out_value_eth,unique_internal_receivers,internal_in_tx,internal_in_value_eth,unique_internal_senders,incoming_from_external_tx,incoming_from_external_value_eth,unique_external_senders,outgoing_to_external_tx,outgoing_to_external_value_eth,unique_external_receivers,total_tx_boundary,total_value_eth_boundary,unique_receivers_total,unique_senders_total,unique_counterparties_sum
str,str,str,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,u32
"""phishing_390""","""0x000bf4254dbaf6acb4809ec1511a…","""normal""",1,0.000155,1,1,0.000168,1,0,0.0,0,1,0.0,1,3,0.000323,2,1,3
"""phishing_390""","""0x000d55e4bbb6b8cdff19ef2dec1a…","""normal""",1,0.000145,1,1,0.000151,1,0,0.0,0,1,0.0,1,3,0.000296,2,1,3
"""phishing_390""","""0x000da24bda94cc1c59e7b339ab06…","""normal""",2,0.000747,1,3,0.003483,1,0,0.0,0,3,0.0,1,8,0.00423,2,1,3
"""phishing_390""","""0x00136a2e22f8ed17b9a15e800572…","""normal""",2,0.000707,1,2,0.000725,1,0,0.0,0,2,0.0,1,6,0.001432,2,1,3
"""phishing_390""","""0x0014539b3bcd161e60f2086b04af…","""normal""",1,0.000325,1,1,0.000334,1,0,0.0,0,1,0.0,1,3,0.000659,2,1,3
"""phishing_390""","""0x0015050f65858bc4de26d111c1b9…","""normal""",1,0.000261,1,1,0.000269,1,0,0.0,0,1,0.0,1,3,0.00053,2,1,3
"""phishing_390""","""0x0017621870362696c484f205960f…","""normal""",0,0.0,0,1,0.015773,1,0,0.0,0,0,0.0,0,1,0.015773,0,1,1
"""phishing_390""","""0x00195f521852d0cf848a39645a73…","""normal""",1,0.005968,1,0,0.0,0,1,0.00597,1,0,0.0,0,2,0.011937,1,1,2
"""phishing_390""","""0x0023c0efd0d4fd342236bfea2a7b…","""normal""",1,0.006549,1,0,0.0,0,0,0.0,0,1,0.0,1,2,0.006549,2,0,2


mixer_242
features shape: (5901, 20)


community,address,label,internal_out_tx,internal_out_value_eth,unique_internal_receivers,internal_in_tx,internal_in_value_eth,unique_internal_senders,incoming_from_external_tx,incoming_from_external_value_eth,unique_external_senders,outgoing_to_external_tx,outgoing_to_external_value_eth,unique_external_receivers,total_tx_boundary,total_value_eth_boundary,unique_receivers_total,unique_senders_total,unique_counterparties_sum
str,str,str,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,u32
"""mixer_242""","""0x00024b0cfb6c7ae68408c32e1513…","""normal""",1,0.602284,1,0,0.0,0,1,0.29083,1,0,0.0,0,2,0.893114,1,1,2
"""mixer_242""","""0x001a4c4904ada7acbef177b80859…","""normal""",0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0
"""mixer_242""","""0x0044fb6b14bc24d45dc82f10913e…","""normal""",0,0.0,0,1,0.002198,1,0,0.0,0,1,0.0,1,2,0.002198,1,1,2
"""mixer_242""","""0x00474678499082d85c07d4b5b07c…","""normal""",0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0
"""mixer_242""","""0x0058cdc8ba2dfa609d266110d202…","""normal""",0,0.0,0,1,0.0028,1,0,0.0,0,4,0.000075,4,5,0.002875,4,1,5
"""mixer_242""","""0x006df42722a3573e26a584e587d1…","""normal""",2,457.070842,1,0,0.0,0,2,457.008129,1,0,0.0,0,4,914.078971,1,1,2
"""mixer_242""","""0x007155d9b9dac297b281413f430c…","""normal""",0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0
"""mixer_242""","""0x009fbd14f4129c12b0cc96400967…","""normal""",0,0.0,0,1,0.0018,1,0,0.0,0,0,0.0,0,1,0.0018,0,1,1
"""mixer_242""","""0x00a4aef2b82dc4b6ea84431a55b0…","""normal""",0,0.0,0,1,0.024631,1,0,0.0,0,0,0.0,0,1,0.024631,0,1,1


community,address,label,internal_out_tx,internal_out_value_eth,unique_internal_receivers,internal_in_tx,internal_in_value_eth,unique_internal_senders,incoming_from_external_tx,incoming_from_external_value_eth,unique_external_senders,outgoing_to_external_tx,outgoing_to_external_value_eth,unique_external_receivers,total_tx_boundary,total_value_eth_boundary,unique_receivers_total,unique_senders_total,unique_counterparties_sum
str,str,str,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,f64,u32,u32,u32
"""mixer_242""","""0x28c6c06298d514db089934071355…","""fraud""",13362,67953.085619,2086,1735,11559.800424,1469,3240,231381.69608,1590,17093,209811.987347,2819,35430,520706.569469,4905,3059,7964
"""mixer_242""","""0xdfd5293d8e347dfe59e90efd55b2…","""normal""",186,8837.393668,133,3,49748.337012,1,1,0.0,1,17716,53203.425957,4644,17906,111789.156638,4777,2,4779
"""mixer_242""","""0xee7ae85f2fe2239e27d9c1e23fff…","""normal""",0,0.0,0,11167,0.0,1,3,0.000475,2,0,0.0,0,11170,0.000475,0,3,3
"""mixer_242""","""0x699ee12a1d97437a4a1e87c71e5d…","""normal""",0,0.0,0,1,13.65545,1,2,304.881094,1,2785,317.537144,3,2788,636.073688,3,2,5
"""mixer_242""","""0x07ae8551be970cb1cca11dd7a11f…","""normal""",4,0.0,2,0,0.0,0,2,561.365328,2,2297,561.282584,13,2303,1122.647912,15,2,17
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""wash_trading_846""","""0x8cbdc87d6d98b9eab82271efade8…","""normal""",0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0
"""wash_trading_846""","""0xe47762a37f31a0b46d694f6e2716…","""normal""",0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0
"""wash_trading_846""","""0xd910f99686f6b26dec76c34a910b…","""normal""",0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0.0,0,0,0


In [30]:
nodes_communities.write_csv("nodes_communities_base_features.csv")

## 7. Temporal Motif Counting Functions

In [31]:
delta = 3600

communities = [
    "wash_trading_846",
    "phishing_390",
    "mixer_242"
]

### 7.1 Repeated Same Direction Motif

In [32]:
def count_repeated_pairs_in_group(group, delta=3600):
    group = group.sort("timestamp")

    timestamps = group["timestamp"].to_list()
    values = group["value_eth"].to_list()

    left = 0
    pair_count = 0
    motif_value_sum = 0.0
    window_value_sum = 0.0

    for right in range(len(timestamps)):
        current_time = timestamps[right]
        current_value = values[right]

        while current_time - timestamps[left] > delta:
            window_value_sum -= values[left]
            left += 1

        previous_count = right - left

        pair_count += previous_count
        motif_value_sum += window_value_sum + previous_count * current_value

        window_value_sum += current_value

    return pl.DataFrame({
        "source": [group["source"][0]],
        "target": [group["target"][0]],
        "repeat_same_direction_pair_count_1h": [pair_count],
        "repeat_same_direction_pair_value_eth_1h": [motif_value_sum]
    })

In [33]:
def compute_repeated_same_direction(
    community,
    tx,
    nodes_communities,
    delta=3600
):
    print("=" * 80)
    print(f"Processing: {community}")

    tx = (
        tx
        .with_columns([
            pl.col("source").cast(pl.Utf8),
            pl.col("target").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("value_eth").cast(pl.Float64)
        ])
        .select(["source", "target", "timestamp", "value_eth"])
        .sort(["source", "target", "timestamp"])
    )

    base = nodes_communities.filter(pl.col("community") == community)

    print(f"Boundary transactions: {tx.height:,}")
    print(f"Base community nodes: {base.height:,}")

    dyad_features = (
        tx
        .group_by(["source", "target"])
        .map_groups(lambda group: count_repeated_pairs_in_group(group, delta=delta))
        .filter(pl.col("repeat_same_direction_pair_count_1h") > 0)
    )

    print(f"Dyads with repeated same-direction motifs: {dyad_features.height:,}")

    if dyad_features.height > 0:
        print(
            "Total repeated same-direction pairs:",
            dyad_features["repeat_same_direction_pair_count_1h"].sum()
        )
    else:
        print("Total repeated same-direction pairs: 0")

    sender_features = (
        dyad_features
        .group_by("source")
        .agg([
            pl.col("repeat_same_direction_pair_count_1h")
            .sum()
            .alias("repeat_same_direction_as_sender_count_1h"),

            pl.col("repeat_same_direction_pair_value_eth_1h")
            .sum()
            .alias("repeat_same_direction_as_sender_value_eth_1h")
        ])
        .rename({"source": "address"})
    )

    receiver_features = (
        dyad_features
        .group_by("target")
        .agg([
            pl.col("repeat_same_direction_pair_count_1h")
            .sum()
            .alias("repeat_same_direction_as_receiver_count_1h"),

            pl.col("repeat_same_direction_pair_value_eth_1h")
            .sum()
            .alias("repeat_same_direction_as_receiver_value_eth_1h")
        ])
        .rename({"target": "address"})
    )

    node_features = (
        base
        .select(["community", "address", "label", "total_tx_boundary"])
        .join(sender_features, on="address", how="left")
        .join(receiver_features, on="address", how="left")
        .with_columns([
            pl.col("repeat_same_direction_as_sender_count_1h").fill_null(0),
            pl.col("repeat_same_direction_as_sender_value_eth_1h").fill_null(0),
            pl.col("repeat_same_direction_as_receiver_count_1h").fill_null(0),
            pl.col("repeat_same_direction_as_receiver_value_eth_1h").fill_null(0)
        ])
        .with_columns([
            (
                pl.col("repeat_same_direction_as_sender_count_1h") +
                pl.col("repeat_same_direction_as_receiver_count_1h")
            ).alias("repeat_same_direction_total_count_1h"),

            (
                pl.col("repeat_same_direction_as_sender_value_eth_1h") +
                pl.col("repeat_same_direction_as_receiver_value_eth_1h")
            ).alias("repeat_same_direction_total_value_eth_1h")
        ])
        .with_columns([
            pl.when(pl.col("total_tx_boundary") > 0)
            .then(pl.col("repeat_same_direction_total_count_1h") / pl.col("total_tx_boundary"))
            .otherwise(0)
            .alias("repeat_same_direction_per_tx_1h")
        ])
    )

    summary = (
        node_features
        .group_by("label")
        .agg([
            pl.len().alias("n_nodes"),

            (pl.col("repeat_same_direction_total_count_1h") > 0)
            .sum()
            .alias("active_repeat_nodes"),

            (
                (pl.col("repeat_same_direction_total_count_1h") > 0).sum() / pl.len()
            ).alias("active_repeat_share"),

            pl.col("repeat_same_direction_total_count_1h").mean().alias("mean_repeat_total_count_1h"),
            pl.col("repeat_same_direction_total_count_1h").median().alias("median_repeat_total_count_1h"),
            pl.col("repeat_same_direction_total_count_1h").max().alias("max_repeat_total_count_1h"),

            pl.col("repeat_same_direction_as_sender_count_1h").mean().alias("mean_repeat_as_sender_1h"),
            pl.col("repeat_same_direction_as_receiver_count_1h").mean().alias("mean_repeat_as_receiver_1h"),

            pl.col("repeat_same_direction_per_tx_1h").mean().alias("mean_repeat_per_tx_1h"),
            pl.col("repeat_same_direction_per_tx_1h").median().alias("median_repeat_per_tx_1h"),

            pl.col("repeat_same_direction_total_value_eth_1h").mean().alias("mean_repeat_value_eth_1h"),
            pl.col("repeat_same_direction_total_value_eth_1h").median().alias("median_repeat_value_eth_1h")
        ])
        .with_columns(pl.lit(community).alias("community"))
        .select([
            "community",
            "label",
            "n_nodes",
            "active_repeat_nodes",
            "active_repeat_share",
            "mean_repeat_total_count_1h",
            "median_repeat_total_count_1h",
            "max_repeat_total_count_1h",
            "mean_repeat_as_sender_1h",
            "mean_repeat_as_receiver_1h",
            "mean_repeat_per_tx_1h",
            "median_repeat_per_tx_1h",
            "mean_repeat_value_eth_1h",
            "median_repeat_value_eth_1h"
        ])
        .sort("label")
    )

    return node_features, summary

### 7.2 Reciprocity Motif

In [34]:
def compute_reciprocity_for_community(
    community,
    tx,
    nodes_communities,
    delta=3600
):
    print("=" * 80)
    print(f"Processing reciprocity: {community}")

    tx = (
        tx
        .with_columns([
            pl.col("hash").cast(pl.Utf8),
            pl.col("source").cast(pl.Utf8),
            pl.col("target").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("value_eth").cast(pl.Float64)
        ])
        .select(["hash", "source", "target", "timestamp", "value_eth"])
        .sort(["source", "target", "timestamp"])
    )

    base = nodes_communities.filter(pl.col("community") == community)

    print(f"Boundary transactions: {tx.height:,}")
    print(f"Base community nodes: {base.height:,}")

    tx_first = tx.select([
        pl.col("hash").alias("hash_1"),
        pl.col("source").alias("source_1"),
        pl.col("target").alias("target_1"),
        pl.col("timestamp").alias("timestamp_1"),
        pl.col("value_eth").alias("value_eth_1")
    ])

    tx_second = tx.select([
        pl.col("hash").alias("hash_2"),
        pl.col("source").alias("source_2"),
        pl.col("target").alias("target_2"),
        pl.col("timestamp").alias("timestamp_2"),
        pl.col("value_eth").alias("value_eth_2")
    ])

    reciprocity_pairs = (
        tx_first
        .join(
            tx_second,
            left_on=["source_1", "target_1"],
            right_on=["target_2", "source_2"],
            how="inner"
        )
        .filter(
            (pl.col("timestamp_2") > pl.col("timestamp_1")) &
            ((pl.col("timestamp_2") - pl.col("timestamp_1")) <= delta)
        )
        .with_columns([
            (pl.col("timestamp_2") - pl.col("timestamp_1")).alias("delta_seconds"),
            (pl.col("value_eth_1") + pl.col("value_eth_2")).alias("motif_value_eth")
        ])
    )

    print(f"Reciprocity pairs: {reciprocity_pairs.height:,}")

    first_sender_features = (
        reciprocity_pairs
        .group_by("source_1")
        .agg([
            pl.len().alias("reciprocity_as_first_sender_count_1h"),
            pl.col("motif_value_eth").sum().alias("reciprocity_as_first_sender_value_eth_1h")
        ])
        .rename({"source_1": "address"})
    )

    first_receiver_features = (
        reciprocity_pairs
        .group_by("target_1")
        .agg([
            pl.len().alias("reciprocity_as_first_receiver_count_1h"),
            pl.col("motif_value_eth").sum().alias("reciprocity_as_first_receiver_value_eth_1h")
        ])
        .rename({"target_1": "address"})
    )

    node_features = (
        base
        .select(["community", "address", "label", "total_tx_boundary"])
        .join(first_sender_features, on="address", how="left")
        .join(first_receiver_features, on="address", how="left")
        .with_columns([
            pl.col("reciprocity_as_first_sender_count_1h").fill_null(0),
            pl.col("reciprocity_as_first_sender_value_eth_1h").fill_null(0),
            pl.col("reciprocity_as_first_receiver_count_1h").fill_null(0),
            pl.col("reciprocity_as_first_receiver_value_eth_1h").fill_null(0)
        ])
        .with_columns([
            (
                pl.col("reciprocity_as_first_sender_count_1h") +
                pl.col("reciprocity_as_first_receiver_count_1h")
            ).alias("reciprocity_total_count_1h"),

            (
                pl.col("reciprocity_as_first_sender_value_eth_1h") +
                pl.col("reciprocity_as_first_receiver_value_eth_1h")
            ).alias("reciprocity_total_value_eth_1h")
        ])
        .with_columns([
            pl.when(pl.col("total_tx_boundary") > 0)
            .then(pl.col("reciprocity_total_count_1h") / pl.col("total_tx_boundary"))
            .otherwise(0)
            .alias("reciprocity_per_tx_1h")
        ])
    )

    summary = (
        node_features
        .group_by("label")
        .agg([
            pl.len().alias("n_nodes"),

            (pl.col("reciprocity_total_count_1h") > 0)
            .sum()
            .alias("active_reciprocity_nodes"),

            (
                (pl.col("reciprocity_total_count_1h") > 0).sum() / pl.len()
            ).alias("active_reciprocity_share"),

            pl.col("reciprocity_total_count_1h").mean().alias("mean_reciprocity_total_count_1h"),
            pl.col("reciprocity_total_count_1h").median().alias("median_reciprocity_total_count_1h"),
            pl.col("reciprocity_total_count_1h").max().alias("max_reciprocity_total_count_1h"),

            pl.col("reciprocity_as_first_sender_count_1h").mean().alias("mean_reciprocity_as_first_sender_1h"),
            pl.col("reciprocity_as_first_receiver_count_1h").mean().alias("mean_reciprocity_as_first_receiver_1h"),

            pl.col("reciprocity_per_tx_1h").mean().alias("mean_reciprocity_per_tx_1h"),
            pl.col("reciprocity_per_tx_1h").median().alias("median_reciprocity_per_tx_1h"),

            pl.col("reciprocity_total_value_eth_1h").mean().alias("mean_reciprocity_value_eth_1h"),
            pl.col("reciprocity_total_value_eth_1h").median().alias("median_reciprocity_value_eth_1h")
        ])
        .with_columns(pl.lit(community).alias("community"))
        .select([
            "community",
            "label",
            "n_nodes",
            "active_reciprocity_nodes",
            "active_reciprocity_share",
            "mean_reciprocity_total_count_1h",
            "median_reciprocity_total_count_1h",
            "max_reciprocity_total_count_1h",
            "mean_reciprocity_as_first_sender_1h",
            "mean_reciprocity_as_first_receiver_1h",
            "mean_reciprocity_per_tx_1h",
            "median_reciprocity_per_tx_1h",
            "mean_reciprocity_value_eth_1h",
            "median_reciprocity_value_eth_1h"
        ])
        .sort("label")
    )

    return node_features, summary

### 7.3 Chain Motif

In [35]:
def empty_chain_contribs():
    return pl.DataFrame({
        "address": pl.Series([], dtype=pl.Utf8),
        "chain_as_start_count_1h": pl.Series([], dtype=pl.Int64),
        "chain_as_start_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "chain_as_middle_count_1h": pl.Series([], dtype=pl.Int64),
        "chain_as_middle_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "chain_as_end_count_1h": pl.Series([], dtype=pl.Int64),
        "chain_as_end_value_eth_1h": pl.Series([], dtype=pl.Float64)
    })


def count_chain_contribs_for_middle(group, delta=3600):
    middle = group["middle"][0]

    incoming = (
        group
        .filter(pl.col("direction") == "incoming")
        .sort("timestamp")
    )

    outgoing = (
        group
        .filter(pl.col("direction") == "outgoing")
        .sort("timestamp")
    )

    if incoming.height == 0 or outgoing.height == 0:
        return empty_chain_contribs()

    in_sources = incoming["counterparty"].to_list()
    in_times = incoming["timestamp"].to_list()
    in_values = incoming["value_eth"].to_list()

    out_targets = outgoing["counterparty"].to_list()
    out_times = outgoing["timestamp"].to_list()
    out_values = outgoing["value_eth"].to_list()

    contrib = defaultdict(lambda: [0, 0.0, 0, 0.0, 0, 0.0])

    for i in range(len(in_times)):
        a = in_sources[i]
        t_in = in_times[i]
        v_in = in_values[i]

        left = bisect_right(out_times, t_in)
        right = bisect_right(out_times, t_in + delta)

        for j in range(left, right):
            c = out_targets[j]

            if a == middle or middle == c or a == c:
                continue

            v_out = out_values[j]
            motif_value = v_in + v_out

            contrib[a][0] += 1
            contrib[a][1] += motif_value

            contrib[middle][2] += 1
            contrib[middle][3] += motif_value

            contrib[c][4] += 1
            contrib[c][5] += motif_value

    if len(contrib) == 0:
        return empty_chain_contribs()

    rows = []

    for address, values in contrib.items():
        rows.append((
            address,
            values[0],
            values[1],
            values[2],
            values[3],
            values[4],
            values[5]
        ))

    return pl.DataFrame(
        rows,
        schema=[
            "address",
            "chain_as_start_count_1h",
            "chain_as_start_value_eth_1h",
            "chain_as_middle_count_1h",
            "chain_as_middle_value_eth_1h",
            "chain_as_end_count_1h",
            "chain_as_end_value_eth_1h"
        ],
        orient="row"
    )

In [36]:
def compute_chain_for_community(
    community,
    tx,
    nodes_communities,
    delta=3600
):
    print("=" * 80)
    print(f"Processing chain: {community}")

    tx = (
        tx
        .with_columns([
            pl.col("source").cast(pl.Utf8),
            pl.col("target").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("value_eth").cast(pl.Float64)
        ])
        .select(["source", "target", "timestamp", "value_eth"])
        .filter(pl.col("source") != pl.col("target"))
    )

    base = nodes_communities.filter(pl.col("community") == community)

    print(f"Boundary transactions: {tx.height:,}")
    print(f"Base community nodes: {base.height:,}")

    incoming_events = (
        tx
        .select([
            pl.col("target").alias("middle"),
            pl.col("source").alias("counterparty"),
            pl.col("timestamp"),
            pl.col("value_eth")
        ])
        .with_columns(pl.lit("incoming").alias("direction"))
    )

    outgoing_events = (
        tx
        .select([
            pl.col("source").alias("middle"),
            pl.col("target").alias("counterparty"),
            pl.col("timestamp"),
            pl.col("value_eth")
        ])
        .with_columns(pl.lit("outgoing").alias("direction"))
    )

    events = pl.concat([incoming_events, outgoing_events])

    chain_contribs_raw = (
        events
        .group_by("middle")
        .map_groups(lambda group: count_chain_contribs_for_middle(group, delta=delta))
    )

    if chain_contribs_raw.height == 0:
        chain_features = empty_chain_contribs()
    else:
        chain_features = (
            chain_contribs_raw
            .group_by("address")
            .agg([
                pl.col("chain_as_start_count_1h").sum(),
                pl.col("chain_as_start_value_eth_1h").sum(),
                pl.col("chain_as_middle_count_1h").sum(),
                pl.col("chain_as_middle_value_eth_1h").sum(),
                pl.col("chain_as_end_count_1h").sum(),
                pl.col("chain_as_end_value_eth_1h").sum()
            ])
        )

    print(f"Nodes participating in chain motifs: {chain_features.height:,}")

    node_features = (
        base
        .select(["community", "address", "label", "total_tx_boundary"])
        .join(chain_features, on="address", how="left")
        .with_columns([
            pl.col("chain_as_start_count_1h").fill_null(0),
            pl.col("chain_as_start_value_eth_1h").fill_null(0),
            pl.col("chain_as_middle_count_1h").fill_null(0),
            pl.col("chain_as_middle_value_eth_1h").fill_null(0),
            pl.col("chain_as_end_count_1h").fill_null(0),
            pl.col("chain_as_end_value_eth_1h").fill_null(0)
        ])
        .with_columns([
            (
                pl.col("chain_as_start_count_1h") +
                pl.col("chain_as_middle_count_1h") +
                pl.col("chain_as_end_count_1h")
            ).alias("chain_total_count_1h"),

            (
                pl.col("chain_as_start_value_eth_1h") +
                pl.col("chain_as_middle_value_eth_1h") +
                pl.col("chain_as_end_value_eth_1h")
            ).alias("chain_total_value_eth_1h")
        ])
        .with_columns([
            pl.when(pl.col("total_tx_boundary") > 0)
            .then(pl.col("chain_total_count_1h") / pl.col("total_tx_boundary"))
            .otherwise(0)
            .alias("chain_per_tx_1h")
        ])
    )

    summary = (
        node_features
        .group_by("label")
        .agg([
            pl.len().alias("n_nodes"),

            (pl.col("chain_total_count_1h") > 0)
            .sum()
            .alias("active_chain_nodes"),

            ((pl.col("chain_total_count_1h") > 0).sum() / pl.len())
            .alias("active_chain_share"),

            pl.col("chain_total_count_1h").mean().alias("mean_chain_total_count_1h"),
            pl.col("chain_total_count_1h").median().alias("median_chain_total_count_1h"),
            pl.col("chain_total_count_1h").max().alias("max_chain_total_count_1h"),

            pl.col("chain_as_start_count_1h").mean().alias("mean_chain_as_start_1h"),
            pl.col("chain_as_middle_count_1h").mean().alias("mean_chain_as_middle_1h"),
            pl.col("chain_as_end_count_1h").mean().alias("mean_chain_as_end_1h"),

            pl.col("chain_per_tx_1h").mean().alias("mean_chain_per_tx_1h"),
            pl.col("chain_per_tx_1h").median().alias("median_chain_per_tx_1h"),

            pl.col("chain_total_value_eth_1h").mean().alias("mean_chain_value_eth_1h"),
            pl.col("chain_total_value_eth_1h").median().alias("median_chain_value_eth_1h")
        ])
        .with_columns(pl.lit(community).alias("community"))
        .select([
            "community",
            "label",
            "n_nodes",
            "active_chain_nodes",
            "active_chain_share",
            "mean_chain_total_count_1h",
            "median_chain_total_count_1h",
            "max_chain_total_count_1h",
            "mean_chain_as_start_1h",
            "mean_chain_as_middle_1h",
            "mean_chain_as_end_1h",
            "mean_chain_per_tx_1h",
            "median_chain_per_tx_1h",
            "mean_chain_value_eth_1h",
            "median_chain_value_eth_1h"
        ])
        .sort("label")
    )

    print(f"Rows: {node_features.height:,}")
    print(f"Columns: {len(node_features.columns):,}")

    return node_features, summary

### 7.4 Fan-In Motif

In [37]:
def empty_fan_in_contribs():
    return pl.DataFrame({
        "address": pl.Series([], dtype=pl.Utf8),
        "fan_in_as_sender_count_1h": pl.Series([], dtype=pl.Int64),
        "fan_in_as_sender_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "fan_in_as_center_count_1h": pl.Series([], dtype=pl.Int64),
        "fan_in_as_center_value_eth_1h": pl.Series([], dtype=pl.Float64)
    })


def count_fan_in_contribs_for_center(group, delta=3600):
    group = group.sort("timestamp")

    center = group["target"][0]
    sources = group["source"].to_list()
    times = group["timestamp"].to_list()
    values = group["value_eth"].to_list()

    contrib = defaultdict(lambda: [0, 0.0, 0, 0.0])

    left = 0
    window_count = 0
    window_value_sum = 0.0
    source_counts = defaultdict(int)
    source_value_sums = defaultdict(float)

    for right in range(len(times)):
        current_time = times[right]
        current_source = sources[right]
        current_value = values[right]

        while current_time - times[left] > delta:
            old_source = sources[left]
            old_value = values[left]

            window_count -= 1
            window_value_sum -= old_value
            source_counts[old_source] -= 1
            source_value_sums[old_source] -= old_value

            if source_counts[old_source] == 0:
                del source_counts[old_source]
                del source_value_sums[old_source]

            left += 1

        same_source_count = source_counts.get(current_source, 0)
        same_source_value_sum = source_value_sums.get(current_source, 0.0)

        valid_previous_count = window_count - same_source_count
        valid_previous_value_sum = window_value_sum - same_source_value_sum

        if valid_previous_count > 0:
            current_sender_value = valid_previous_value_sum + valid_previous_count * current_value

            contrib[current_source][0] += valid_previous_count
            contrib[current_source][1] += current_sender_value

            contrib[center][2] += valid_previous_count
            contrib[center][3] += current_sender_value

            for previous_source, previous_count in source_counts.items():
                if previous_source == current_source:
                    continue

                previous_value_sum = source_value_sums[previous_source]
                previous_sender_value = previous_value_sum + previous_count * current_value

                contrib[previous_source][0] += previous_count
                contrib[previous_source][1] += previous_sender_value

        window_count += 1
        window_value_sum += current_value
        source_counts[current_source] += 1
        source_value_sums[current_source] += current_value

    if len(contrib) == 0:
        return empty_fan_in_contribs()

    rows = []

    for address, values_list in contrib.items():
        rows.append((
            address,
            values_list[0],
            values_list[1],
            values_list[2],
            values_list[3]
        ))

    return pl.DataFrame(
        rows,
        schema=[
            "address",
            "fan_in_as_sender_count_1h",
            "fan_in_as_sender_value_eth_1h",
            "fan_in_as_center_count_1h",
            "fan_in_as_center_value_eth_1h"
        ],
        orient="row"
    )

In [38]:
def compute_fan_in_for_community(
    community,
    tx,
    nodes_communities,
    delta=3600
):
    print("=" * 80)
    print(f"Processing fan-in: {community}")

    tx = (
        tx
        .with_columns([
            pl.col("source").cast(pl.Utf8),
            pl.col("target").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("value_eth").cast(pl.Float64)
        ])
        .select(["source", "target", "timestamp", "value_eth"])
        .filter(pl.col("source") != pl.col("target"))
        .sort(["target", "timestamp"])
    )

    base = nodes_communities.filter(pl.col("community") == community)

    print(f"Boundary transactions: {tx.height:,}")
    print(f"Base community nodes: {base.height:,}")

    fan_in_contribs_raw = (
        tx
        .group_by("target")
        .map_groups(lambda group: count_fan_in_contribs_for_center(group, delta=delta))
    )

    if fan_in_contribs_raw.height == 0:
        fan_in_features = empty_fan_in_contribs()
    else:
        fan_in_features = (
            fan_in_contribs_raw
            .group_by("address")
            .agg([
                pl.col("fan_in_as_sender_count_1h").sum(),
                pl.col("fan_in_as_sender_value_eth_1h").sum(),
                pl.col("fan_in_as_center_count_1h").sum(),
                pl.col("fan_in_as_center_value_eth_1h").sum()
            ])
        )

    print(f"Nodes participating in fan-in motifs: {fan_in_features.height:,}")

    node_features = (
        base
        .select(["community", "address", "label", "total_tx_boundary"])
        .join(fan_in_features, on="address", how="left")
        .with_columns([
            pl.col("fan_in_as_sender_count_1h").fill_null(0),
            pl.col("fan_in_as_sender_value_eth_1h").fill_null(0),
            pl.col("fan_in_as_center_count_1h").fill_null(0),
            pl.col("fan_in_as_center_value_eth_1h").fill_null(0)
        ])
        .with_columns([
            (
                pl.col("fan_in_as_sender_count_1h") +
                pl.col("fan_in_as_center_count_1h")
            ).alias("fan_in_total_count_1h"),

            (
                pl.col("fan_in_as_sender_value_eth_1h") +
                pl.col("fan_in_as_center_value_eth_1h")
            ).alias("fan_in_total_value_eth_1h")
        ])
        .with_columns([
            pl.when(pl.col("total_tx_boundary") > 0)
            .then(pl.col("fan_in_total_count_1h") / pl.col("total_tx_boundary"))
            .otherwise(0)
            .alias("fan_in_per_tx_1h")
        ])
    )

    summary = (
        node_features
        .group_by("label")
        .agg([
            pl.len().alias("n_nodes"),

            (pl.col("fan_in_total_count_1h") > 0)
            .sum()
            .alias("active_fan_in_nodes"),

            ((pl.col("fan_in_total_count_1h") > 0).sum() / pl.len())
            .alias("active_fan_in_share"),

            pl.col("fan_in_total_count_1h").mean().alias("mean_fan_in_total_count_1h"),
            pl.col("fan_in_total_count_1h").median().alias("median_fan_in_total_count_1h"),
            pl.col("fan_in_total_count_1h").max().alias("max_fan_in_total_count_1h"),

            pl.col("fan_in_as_sender_count_1h").mean().alias("mean_fan_in_as_sender_1h"),
            pl.col("fan_in_as_center_count_1h").mean().alias("mean_fan_in_as_center_1h"),

            pl.col("fan_in_per_tx_1h").mean().alias("mean_fan_in_per_tx_1h"),
            pl.col("fan_in_per_tx_1h").median().alias("median_fan_in_per_tx_1h"),

            pl.col("fan_in_total_value_eth_1h").mean().alias("mean_fan_in_value_eth_1h"),
            pl.col("fan_in_total_value_eth_1h").median().alias("median_fan_in_value_eth_1h")
        ])
        .with_columns(pl.lit(community).alias("community"))
        .select([
            "community",
            "label",
            "n_nodes",
            "active_fan_in_nodes",
            "active_fan_in_share",
            "mean_fan_in_total_count_1h",
            "median_fan_in_total_count_1h",
            "max_fan_in_total_count_1h",
            "mean_fan_in_as_sender_1h",
            "mean_fan_in_as_center_1h",
            "mean_fan_in_per_tx_1h",
            "median_fan_in_per_tx_1h",
            "mean_fan_in_value_eth_1h",
            "median_fan_in_value_eth_1h"
        ])
        .sort("label")
    )

    print(f"Rows: {node_features.height:,}")
    print(f"Columns: {len(node_features.columns):,}")

    return node_features, summary

### 7.5 Fan-Out Motif

In [39]:
def empty_fan_out_contribs():
    return pl.DataFrame({
        "address": pl.Series([], dtype=pl.Utf8),
        "fan_out_as_center_count_1h": pl.Series([], dtype=pl.Int64),
        "fan_out_as_center_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "fan_out_as_receiver_count_1h": pl.Series([], dtype=pl.Int64),
        "fan_out_as_receiver_value_eth_1h": pl.Series([], dtype=pl.Float64)
    })


def count_fan_out_contribs_for_center(group, delta=3600):
    group = group.sort("timestamp")

    center = group["source"][0]
    targets = group["target"].to_list()
    times = group["timestamp"].to_list()
    values = group["value_eth"].to_list()

    contrib = defaultdict(lambda: [0, 0.0, 0, 0.0])

    left = 0
    window_count = 0
    window_value_sum = 0.0
    target_counts = defaultdict(int)
    target_value_sums = defaultdict(float)

    for right in range(len(times)):
        current_time = times[right]
        current_target = targets[right]
        current_value = values[right]

        while current_time - times[left] > delta:
            old_target = targets[left]
            old_value = values[left]

            window_count -= 1
            window_value_sum -= old_value
            target_counts[old_target] -= 1
            target_value_sums[old_target] -= old_value

            if target_counts[old_target] == 0:
                del target_counts[old_target]
                del target_value_sums[old_target]

            left += 1

        same_target_count = target_counts.get(current_target, 0)
        same_target_value_sum = target_value_sums.get(current_target, 0.0)

        valid_previous_count = window_count - same_target_count
        valid_previous_value_sum = window_value_sum - same_target_value_sum

        if valid_previous_count > 0:
            current_receiver_value = valid_previous_value_sum + valid_previous_count * current_value

            contrib[center][0] += valid_previous_count
            contrib[center][1] += current_receiver_value

            contrib[current_target][2] += valid_previous_count
            contrib[current_target][3] += current_receiver_value

            for previous_target, previous_count in target_counts.items():
                if previous_target == current_target:
                    continue

                previous_value_sum = target_value_sums[previous_target]
                previous_receiver_value = previous_value_sum + previous_count * current_value

                contrib[previous_target][2] += previous_count
                contrib[previous_target][3] += previous_receiver_value

        window_count += 1
        window_value_sum += current_value
        target_counts[current_target] += 1
        target_value_sums[current_target] += current_value

    if len(contrib) == 0:
        return empty_fan_out_contribs()

    rows = []

    for address, values_list in contrib.items():
        rows.append((
            address,
            values_list[0],
            values_list[1],
            values_list[2],
            values_list[3]
        ))

    return pl.DataFrame(
        rows,
        schema=[
            "address",
            "fan_out_as_center_count_1h",
            "fan_out_as_center_value_eth_1h",
            "fan_out_as_receiver_count_1h",
            "fan_out_as_receiver_value_eth_1h"
        ],
        orient="row"
    )

In [40]:
def compute_fan_out_for_community(
    community,
    tx,
    nodes_communities,
    delta=3600
):
    print("=" * 80)
    print(f"Processing fan-out: {community}")

    tx = (
        tx
        .with_columns([
            pl.col("source").cast(pl.Utf8),
            pl.col("target").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("value_eth").cast(pl.Float64)
        ])
        .select(["source", "target", "timestamp", "value_eth"])
        .filter(pl.col("source") != pl.col("target"))
        .sort(["source", "timestamp"])
    )

    base = nodes_communities.filter(pl.col("community") == community)

    print(f"Boundary transactions: {tx.height:,}")
    print(f"Base community nodes: {base.height:,}")

    fan_out_contribs_raw = (
        tx
        .group_by("source")
        .map_groups(lambda group: count_fan_out_contribs_for_center(group, delta=delta))
    )

    if fan_out_contribs_raw.height == 0:
        fan_out_features = empty_fan_out_contribs()
    else:
        fan_out_features = (
            fan_out_contribs_raw
            .group_by("address")
            .agg([
                pl.col("fan_out_as_center_count_1h").sum(),
                pl.col("fan_out_as_center_value_eth_1h").sum(),
                pl.col("fan_out_as_receiver_count_1h").sum(),
                pl.col("fan_out_as_receiver_value_eth_1h").sum()
            ])
        )

    print(f"Nodes participating in fan-out motifs: {fan_out_features.height:,}")

    node_features = (
        base
        .select(["community", "address", "label", "total_tx_boundary"])
        .join(fan_out_features, on="address", how="left")
        .with_columns([
            pl.col("fan_out_as_center_count_1h").fill_null(0),
            pl.col("fan_out_as_center_value_eth_1h").fill_null(0),
            pl.col("fan_out_as_receiver_count_1h").fill_null(0),
            pl.col("fan_out_as_receiver_value_eth_1h").fill_null(0)
        ])
        .with_columns([
            (
                pl.col("fan_out_as_center_count_1h") +
                pl.col("fan_out_as_receiver_count_1h")
            ).alias("fan_out_total_count_1h"),

            (
                pl.col("fan_out_as_center_value_eth_1h") +
                pl.col("fan_out_as_receiver_value_eth_1h")
            ).alias("fan_out_total_value_eth_1h")
        ])
        .with_columns([
            pl.when(pl.col("total_tx_boundary") > 0)
            .then(pl.col("fan_out_total_count_1h") / pl.col("total_tx_boundary"))
            .otherwise(0)
            .alias("fan_out_per_tx_1h")
        ])
    )

    summary = (
        node_features
        .group_by("label")
        .agg([
            pl.len().alias("n_nodes"),

            (pl.col("fan_out_total_count_1h") > 0)
            .sum()
            .alias("active_fan_out_nodes"),

            ((pl.col("fan_out_total_count_1h") > 0).sum() / pl.len())
            .alias("active_fan_out_share"),

            pl.col("fan_out_total_count_1h").mean().alias("mean_fan_out_total_count_1h"),
            pl.col("fan_out_total_count_1h").median().alias("median_fan_out_total_count_1h"),
            pl.col("fan_out_total_count_1h").max().alias("max_fan_out_total_count_1h"),

            pl.col("fan_out_as_center_count_1h").mean().alias("mean_fan_out_as_center_1h"),
            pl.col("fan_out_as_receiver_count_1h").mean().alias("mean_fan_out_as_receiver_1h"),

            pl.col("fan_out_per_tx_1h").mean().alias("mean_fan_out_per_tx_1h"),
            pl.col("fan_out_per_tx_1h").median().alias("median_fan_out_per_tx_1h"),

            pl.col("fan_out_total_value_eth_1h").mean().alias("mean_fan_out_value_eth_1h"),
            pl.col("fan_out_total_value_eth_1h").median().alias("median_fan_out_value_eth_1h")
        ])
        .with_columns(pl.lit(community).alias("community"))
        .select([
            "community",
            "label",
            "n_nodes",
            "active_fan_out_nodes",
            "active_fan_out_share",
            "mean_fan_out_total_count_1h",
            "median_fan_out_total_count_1h",
            "max_fan_out_total_count_1h",
            "mean_fan_out_as_center_1h",
            "mean_fan_out_as_receiver_1h",
            "mean_fan_out_per_tx_1h",
            "median_fan_out_per_tx_1h",
            "mean_fan_out_value_eth_1h",
            "median_fan_out_value_eth_1h"
        ])
        .sort("label")
    )

    print(f"Rows: {node_features.height:,}")
    print(f"Columns: {len(node_features.columns):,}")

    return node_features, summary

### 7.6 Cycle Motif

In [41]:
def empty_cycle_contribs():
    return pl.DataFrame({
        "address": pl.Series([], dtype=pl.Utf8),
        "cycle_as_start_count_1h": pl.Series([], dtype=pl.Int64),
        "cycle_as_start_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "cycle_as_middle_count_1h": pl.Series([], dtype=pl.Int64),
        "cycle_as_middle_value_eth_1h": pl.Series([], dtype=pl.Float64),
        "cycle_as_end_count_1h": pl.Series([], dtype=pl.Int64),
        "cycle_as_end_value_eth_1h": pl.Series([], dtype=pl.Float64)
    })

def build_cycle_indexes(tx):
    incoming_by_target = defaultdict(lambda: {"times": [], "sources": [], "values": []})
    closing_by_pair = defaultdict(lambda: {"times": [], "values": []})

    tx_sorted = tx.sort("timestamp")

    for source, target, timestamp, value_eth in tx_sorted.iter_rows():
        incoming_by_target[target]["times"].append(timestamp)
        incoming_by_target[target]["sources"].append(source)
        incoming_by_target[target]["values"].append(value_eth)

        closing_by_pair[(source, target)]["times"].append(timestamp)
        closing_by_pair[(source, target)]["values"].append(value_eth)

    closing_prefix_by_pair = {}

    for pair, data in closing_by_pair.items():
        prefix = [0.0]

        for value in data["values"]:
            prefix.append(prefix[-1] + value)

        closing_prefix_by_pair[pair] = prefix

    return incoming_by_target, closing_by_pair, closing_prefix_by_pair

In [42]:
def compute_cycle_contribs(tx, delta=3600):
    incoming_by_target, closing_by_pair, closing_prefix_by_pair = build_cycle_indexes(tx)

    contrib = defaultdict(lambda: [0, 0.0, 0, 0.0, 0, 0.0])

    tx_second_edges = tx.sort("timestamp")

    n_second_edges = tx_second_edges.height

    for idx, row in enumerate(tx_second_edges.iter_rows(named=True)):
        b = row["source"]
        c = row["target"]
        t2 = row["timestamp"]
        v2 = row["value_eth"]

        incoming_data = incoming_by_target.get(b)

        if incoming_data is None:
            continue

        in_times = incoming_data["times"]
        in_sources = incoming_data["sources"]
        in_values = incoming_data["values"]

        left_in = bisect_left(in_times, t2 - delta)
        right_in = bisect_left(in_times, t2)

        if left_in == right_in:
            continue

        for i in range(left_in, right_in):
            a = in_sources[i]
            t1 = in_times[i]
            v1 = in_values[i]

            if a == b or b == c or a == c:
                continue

            closing_pair = (c, a)
            closing_data = closing_by_pair.get(closing_pair)

            if closing_data is None:
                continue

            close_times = closing_data["times"]
            close_prefix = closing_prefix_by_pair[closing_pair]

            left_close = bisect_right(close_times, t2)
            right_close = bisect_right(close_times, t1 + delta)

            k = right_close - left_close

            if k <= 0:
                continue

            closing_value_sum = close_prefix[right_close] - close_prefix[left_close]
            motif_value_sum = k * (v1 + v2) + closing_value_sum

            contrib[a][0] += k
            contrib[a][1] += motif_value_sum

            contrib[b][2] += k
            contrib[b][3] += motif_value_sum

            contrib[c][4] += k
            contrib[c][5] += motif_value_sum

    if len(contrib) == 0:
        return empty_cycle_contribs()

    rows = []

    for address, values in contrib.items():
        rows.append((
            address,
            values[0],
            values[1],
            values[2],
            values[3],
            values[4],
            values[5]
        ))

    return pl.DataFrame(
        rows,
        schema=[
            "address",
            "cycle_as_start_count_1h",
            "cycle_as_start_value_eth_1h",
            "cycle_as_middle_count_1h",
            "cycle_as_middle_value_eth_1h",
            "cycle_as_end_count_1h",
            "cycle_as_end_value_eth_1h"
        ],
        orient="row"
    )

In [43]:
def compute_cycle_for_community(
    community,
    tx,
    nodes_communities,
    delta=3600
):
    print("=" * 80)
    print(f"Processing cycle: {community}")

    tx = (
        tx
        .with_columns([
            pl.col("source").cast(pl.Utf8),
            pl.col("target").cast(pl.Utf8),
            pl.col("timestamp").cast(pl.Int64),
            pl.col("value_eth").cast(pl.Float64)
        ])
        .select(["source", "target", "timestamp", "value_eth"])
        .filter(pl.col("source") != pl.col("target"))
    )

    base = nodes_communities.filter(pl.col("community") == community)

    print(f"Boundary transactions: {tx.height:,}")
    print(f"Base community nodes: {base.height:,}")

    cycle_features = compute_cycle_contribs(tx, delta=delta)

    print(f"Nodes participating in cycle motifs: {cycle_features.height:,}")

    node_features = (
        base
        .select(["community", "address", "label", "total_tx_boundary"])
        .join(cycle_features, on="address", how="left")
        .with_columns([
            pl.col("cycle_as_start_count_1h").fill_null(0),
            pl.col("cycle_as_start_value_eth_1h").fill_null(0),
            pl.col("cycle_as_middle_count_1h").fill_null(0),
            pl.col("cycle_as_middle_value_eth_1h").fill_null(0),
            pl.col("cycle_as_end_count_1h").fill_null(0),
            pl.col("cycle_as_end_value_eth_1h").fill_null(0)
        ])
        .with_columns([
            (
                pl.col("cycle_as_start_count_1h") +
                pl.col("cycle_as_middle_count_1h") +
                pl.col("cycle_as_end_count_1h")
            ).alias("cycle_total_count_1h"),

            (
                pl.col("cycle_as_start_value_eth_1h") +
                pl.col("cycle_as_middle_value_eth_1h") +
                pl.col("cycle_as_end_value_eth_1h")
            ).alias("cycle_total_value_eth_1h")
        ])
        .with_columns([
            pl.when(pl.col("total_tx_boundary") > 0)
            .then(pl.col("cycle_total_count_1h") / pl.col("total_tx_boundary"))
            .otherwise(0)
            .alias("cycle_per_tx_1h")
        ])
    )

    summary = (
        node_features
        .group_by("label")
        .agg([
            pl.len().alias("n_nodes"),

            (pl.col("cycle_total_count_1h") > 0)
            .sum()
            .alias("active_cycle_nodes"),

            ((pl.col("cycle_total_count_1h") > 0).sum() / pl.len())
            .alias("active_cycle_share"),

            pl.col("cycle_total_count_1h").mean().alias("mean_cycle_total_count_1h"),
            pl.col("cycle_total_count_1h").median().alias("median_cycle_total_count_1h"),
            pl.col("cycle_total_count_1h").max().alias("max_cycle_total_count_1h"),

            pl.col("cycle_as_start_count_1h").mean().alias("mean_cycle_as_start_1h"),
            pl.col("cycle_as_middle_count_1h").mean().alias("mean_cycle_as_middle_1h"),
            pl.col("cycle_as_end_count_1h").mean().alias("mean_cycle_as_end_1h"),

            pl.col("cycle_per_tx_1h").mean().alias("mean_cycle_per_tx_1h"),
            pl.col("cycle_per_tx_1h").median().alias("median_cycle_per_tx_1h"),

            pl.col("cycle_total_value_eth_1h").mean().alias("mean_cycle_value_eth_1h"),
            pl.col("cycle_total_value_eth_1h").median().alias("median_cycle_value_eth_1h")
        ])
        .with_columns(pl.lit(community).alias("community"))
        .select([
            "community",
            "label",
            "n_nodes",
            "active_cycle_nodes",
            "active_cycle_share",
            "mean_cycle_total_count_1h",
            "median_cycle_total_count_1h",
            "max_cycle_total_count_1h",
            "mean_cycle_as_start_1h",
            "mean_cycle_as_middle_1h",
            "mean_cycle_as_end_1h",
            "mean_cycle_per_tx_1h",
            "median_cycle_per_tx_1h",
            "mean_cycle_value_eth_1h",
            "median_cycle_value_eth_1h"
        ])
        .sort("label")
    )

    print(f"Rows: {node_features.height:,}")
    print(f"Columns: {len(node_features.columns):,}")

    return node_features, summary

### 7.7 Computing Motifs

In [44]:
def compute_motif_for_all_communities(
    motif_name,
    compute_function,
    feature_cols,
    communities,
    boundary_transactions,
    nodes_communities,
    delta=3600
):
    all_features = []
    all_summaries = []

    print("=" * 100)
    print(f"Computing motif: {motif_name}")

    for community in communities:
        node_features, summary = compute_function(
            community=community,
            tx=boundary_transactions[community],
            nodes_communities=nodes_communities,
            delta=delta
        )

        motif_features = node_features.select(
            ["community", "address"] + feature_cols
        )

        all_features.append(motif_features)
        all_summaries.append(summary)

    features_all = pl.concat(all_features)
    summary_all = pl.concat(all_summaries)

    return features_all, summary_all

In [45]:
def add_motif_features_to_nodes(
    nodes_communities,
    motif_features_all,
    feature_cols
):
    nodes_communities = (
        nodes_communities
        .join(
            motif_features_all,
            on=["community", "address"],
            how="left"
        )
        .with_columns([
            pl.col(col).fill_null(0)
            for col in feature_cols
        ])
    )

    return nodes_communities

## 8. Final Feature Table Creation

In [46]:
motif_configs = {
    "repeated_same_direction": {
        "compute_function": compute_repeated_same_direction,
        "feature_cols": [
            "repeat_same_direction_as_sender_count_1h",
            "repeat_same_direction_as_sender_value_eth_1h",
            "repeat_same_direction_as_receiver_count_1h",
            "repeat_same_direction_as_receiver_value_eth_1h",
            "repeat_same_direction_total_count_1h",
            "repeat_same_direction_total_value_eth_1h",
            "repeat_same_direction_per_tx_1h"
        ]
    },

    "reciprocity": {
        "compute_function": compute_reciprocity_for_community,
        "feature_cols": [
            "reciprocity_as_first_sender_count_1h",
            "reciprocity_as_first_sender_value_eth_1h",
            "reciprocity_as_first_receiver_count_1h",
            "reciprocity_as_first_receiver_value_eth_1h",
            "reciprocity_total_count_1h",
            "reciprocity_total_value_eth_1h",
            "reciprocity_per_tx_1h"
        ]
    },

    "chain": {
        "compute_function": compute_chain_for_community,
        "feature_cols": [
            "chain_as_start_count_1h",
            "chain_as_start_value_eth_1h",
            "chain_as_middle_count_1h",
            "chain_as_middle_value_eth_1h",
            "chain_as_end_count_1h",
            "chain_as_end_value_eth_1h",
            "chain_total_count_1h",
            "chain_total_value_eth_1h",
            "chain_per_tx_1h"
        ]
    },

    "fan_in": {
        "compute_function": compute_fan_in_for_community,
        "feature_cols": [
            "fan_in_as_sender_count_1h",
            "fan_in_as_sender_value_eth_1h",
            "fan_in_as_center_count_1h",
            "fan_in_as_center_value_eth_1h",
            "fan_in_total_count_1h",
            "fan_in_total_value_eth_1h",
            "fan_in_per_tx_1h"
        ]
    },

    "fan_out": {
        "compute_function": compute_fan_out_for_community,
        "feature_cols": [
            "fan_out_as_center_count_1h",
            "fan_out_as_center_value_eth_1h",
            "fan_out_as_receiver_count_1h",
            "fan_out_as_receiver_value_eth_1h",
            "fan_out_total_count_1h",
            "fan_out_total_value_eth_1h",
            "fan_out_per_tx_1h"
        ]
    },

    "cycle": {
        "compute_function": compute_cycle_for_community,
        "feature_cols": [
            "cycle_as_start_count_1h",
            "cycle_as_start_value_eth_1h",
            "cycle_as_middle_count_1h",
            "cycle_as_middle_value_eth_1h",
            "cycle_as_end_count_1h",
            "cycle_as_end_value_eth_1h",
            "cycle_total_count_1h",
            "cycle_total_value_eth_1h",
            "cycle_per_tx_1h"
        ]
    }
}

In [47]:
motif_features_results = {}
motif_summary_results = {}

for motif_name, config in motif_configs.items():
    features_all, summary_all = compute_motif_for_all_communities(
        motif_name=motif_name,
        compute_function=config["compute_function"],
        feature_cols=config["feature_cols"],
        communities=communities,
        boundary_transactions=boundary_transactions,
        nodes_communities=nodes_communities,
        delta=delta
    )

    motif_features_results[motif_name] = features_all
    motif_summary_results[motif_name] = summary_all

    nodes_communities = add_motif_features_to_nodes(
        nodes_communities=nodes_communities,
        motif_features_all=features_all,
        feature_cols=config["feature_cols"]
    )

Computing motif: repeated_same_direction
Processing: wash_trading_846
Boundary transactions: 33,681
Base community nodes: 2,820
Dyads with repeated same-direction motifs: 2,331
Total repeated same-direction pairs: 28132
Processing: phishing_390
Boundary transactions: 65,504
Base community nodes: 17,500
Dyads with repeated same-direction motifs: 1,460
Total repeated same-direction pairs: 619931
Processing: mixer_242
Boundary transactions: 70,569
Base community nodes: 5,901
Dyads with repeated same-direction motifs: 1,643
Total repeated same-direction pairs: 3185282
Computing motif: reciprocity
Processing reciprocity: wash_trading_846
Boundary transactions: 33,681
Base community nodes: 2,820
Reciprocity pairs: 25,207
Processing reciprocity: phishing_390
Boundary transactions: 65,504
Base community nodes: 17,500
Reciprocity pairs: 212
Processing reciprocity: mixer_242
Boundary transactions: 70,569
Base community nodes: 5,901
Reciprocity pairs: 73
Computing motif: chain
Processing chain: w

In [48]:
nodes_communities.write_csv("nodes_communities_with_all_motifs.csv")

print("Saved: nodes_communities_with_all_motifs.csv")
print("Rows:", nodes_communities.height)
print("Columns:", len(nodes_communities.columns))

Saved: nodes_communities_with_all_motifs.csv
Rows: 26221
Columns: 66
